# CALM 

## 0. Setup — Tạo LLM một lần, tất cả agent dùng chung

Chạy cell này trước. Biến `llm` được dùng bởi: PlanningAgent, RSENModule, DataKnowledgeAgent, WildfireQAAgent, SafetyChecker, EvaluatorAgent.

In [36]:
import os
import sys

root = os.path.abspath(os.getcwd())
if root.endswith("calm"):
    pkg_root = os.path.join(root, "src")
else:
    pkg_root = os.path.join(root, "calm", "src")
if os.path.exists(pkg_root) and pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

import calm
print(f"CALM version: {calm.__version__}")

# === LLM duy nhất — tất cả agent dùng chung ===
from langchain_openrouter import ChatOpenRouter
llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    api_key=os.environ.get("OPENROUTER_API_KEY", ""),
    temperature=0.0,
)
# Test
print("LLM test:", llm.invoke("Hi").content[:50] if llm.invoke("Hi").content else "OK")


CALM version: 0.1.0


In [37]:
# llm đã tạo ở cell trên — PlanningAgent, RSEN, QA, Evaluator đều dùng chung llm

## 1. Planning Agent

In [38]:
from calm.agents.planning_agent import PlanningAgent

agent = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = agent.invoke(query)

print("Approved:", result.get("approved"))
print("Plan:")
for i, step in enumerate(result.get("final_output") or [], 1):
    print(f"  {i}. {step}")

Planning generator failed: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}
Traceback (most recent call last):
  File "/home/hungvu/code/khanh/v2/calm-/src/calm/agents/planning_agent.py", line 39, in _generator_node
    resp = self.llm.invoke(msgs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 402, in invoke
    self.generate_prompt(
  File "/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 1123, in generate_prompt
    return self.generate(prompt_messages, stop=stop, callbacks=callbacks, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 933, in generate
    self._generate_with_cache(
  File "/home/hungvu/code/khanh/.venv/lib/python3.12

Approved: False
Plan:


## 2. RSEN — Xác thực dự đoán

In [ ]:
from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore

memory = ChromaMemoryStore(
    collection_name="calm_rsen_memory",
    persist_directory=".chroma",
    k=3,
)
rsen = RSENModule(llm=llm, memory_store=memory, k=3)

result = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={
        "temperature": 35.0,
        "humidity": 0.2,
        "wind_speed": 15,
    },
    spatial_data={
        "fuel_type": "Shrubland",
        "slope": 25,
        "elevation": 500,
    },
)

print("Validation:", result.get("validation_decision"))
print("Rationale:", result.get("final_rationale", "")[:300])

## 3. Wildfire QA Agent

In [ ]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
memory = ChromaMemoryStore(
    collection_name="calm_qa_memory",
    persist_directory=".chroma",
    k=3,
)

tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None}
data_agent = DataKnowledgeAgent(
    llm=llm, tools=tools, memory_store=memory, config={"dedup_check": True}
)
qa = WildfireQAAgent(
    llm=llm,
    data_agent=data_agent,
    web_search_tool=web,
    memory_store=memory,
    config={"evidence_threshold": 0.65},
)

result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = result.get("final_output") or {}
print("Answer:", out.get("answer", out))
print("Citations:", out.get("citations", []))

## 4. Full Pipeline — Plan + Evaluator

In [ ]:
from calm.agents.evaluator_agent import EvaluatorAgent
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={})
evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})

query = "Assess wildfire risk for California Central Valley next 14 days"
plan_result = planner.invoke(query)
plan = plan_result.get("final_output") or []

print("Plan steps:", len(plan))
eval_result = evaluator.evaluate(
    response={"plan": plan, "query": query},
    query=query,
)

print("Evaluation passed:", eval_result.get("passed"))
print("Average score:", eval_result.get("average_score"))
print("Scores:", eval_result.get("scores", {}))